# Earth-like configuration

This notebook demonstrates a JAX-ESM (JEM) example using JAX-GCM (JCM), Slab Ocean Model, and Slab Land Model.

In [ ]:
from pathlib import Path
from importlib import resources

import jcm
from jcm.physics.speedy.speedy_coords import get_speedy_coords
from jcm.terrain import TerrainData
from jcm.forcing import ForcingData

import jax_datetime as jdt

from jem.base.coupler import Coupler
from jem.components import (
    JCMComponent,
    SlabLandModel,
    SlabOceanModel,
    SlabSeaiceModel,
)
from jem.components.slab import SlabGrid, SlabLandParameters, SlabOceanParameters

use_ipython = 'get_ipython' in globals()

## Configurations

In [ ]:
start_datetime = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")
simulation_name = "02-01_earth"
output_dir = (Path("output") / simulation_name).resolve()
output_dir.mkdir(exist_ok=True, parents=True)
output_figures = {
    "animation": output_dir / "animation_humidity_sst.gif",
}

In [ ]:
# ## Topography and forcing

coords = get_speedy_coords()  # T31 spectral resolution with 8 vertical levels

# Load realistic orography and land-sea mask, interpolated to T31 grid
data_dir = resources.files("jcm.data.bc.t30.clim")
terrain_file = data_dir / "terrain.nc"
terrain = TerrainData.from_file(terrain_file, coords=coords)

# Load realistic forcing data (SST, sea ice, soil moisture, etc.) interpolated to T31 grid
forcing_file = data_dir / "forcing.nc"
forcing = ForcingData.from_file(forcing_file, coords=coords)

## Creating Flux and Scalar Exchange between Components

In [ ]:
def exchange(components, time):
    """Send the surface heat flux down and the surface state back up.

    An exchanger is traced with the rest of the coupled step, so it builds new
    carries with `.replace(...)` rather than writing into the ones it is handed.
    """
    del time  # this exchange does not depend on the date

    atm = components["atm"]
    ocn = components["ocn"]
    lnd = components["lnd"]

    ocn = dict(ocn, forcing=ocn["forcing"].replace(
        total_heat_flux=atm["derived"].total_heat_flux,
    ))
    lnd = dict(lnd, forcing=lnd["forcing"].replace(
        total_heat_flux=atm["derived"].total_heat_flux,
    ))
    atm = dict(atm, forcing=atm["forcing"].replace(
        sea_surface_temperature=ocn["state"].sea_surface_temperature,
        stl_am=lnd["state"].land_surface_temperature,
        snowc_am=lnd["state"].snowc,
        soilw_am=lnd["state"].soilw,
    ))

    return dict(components, atm=atm, ocn=ocn, lnd=lnd)

## Create Components

In [ ]:
atm_model = jcm.model.Model(
    coords=coords,
    start_date=start_datetime,
    terrain=terrain,
)

# The slab models run on the atmosphere's own horizontal grid, with the land
# fraction the atmosphere was built with -- one source of truth for both.
earth_grid = SlabGrid.from_coords(
    coords.horizontal,
    fractional_mask=terrain.fmask,
)

model = Coupler(
    dict(
        atm=JCMComponent(atm_model),
        ocn=SlabOceanModel(
            earth_grid,
            SlabOceanParameters(
                forcing_method="relaxation",
                relaxation_time=30 * 86400.0,
            ),
            sst_clim_file=str(forcing_file),
        ),
        lnd=SlabLandModel(
            earth_grid,
            # Strong relaxation because the land model seems to have a bug such
            # that temperature gets unrealistically cold/hot.
            SlabLandParameters(tdland=86400.0),
            land_clim_file=str(forcing_file),
        ),
        seaice=SlabSeaiceModel(earth_grid, name="seaice"),
    ),
    dict(exchange=exchange),
    coupling_timestep=coupling_timestep,
    start_date=start_datetime,
)

print(repr(model))

## Run Coupled Model

In [ ]:
simulation_interval = jdt.to_timedelta(30, "day")
run = model.generate_trajectory_function(
    int(simulation_interval / coupling_timestep)
)
initial_carry = model.initialize()
final_carry, diagnostics = run(initial_carry)

## Output into NetCDF

In [ ]:
output_dict = model.to_xarray(diagnostics)
output_dict_subsample = {}
subsample_skip = 5
for component_name, ds in output_dict.items():
    output_file = output_dir / f"{component_name:s}.nc"
    print(f"Output file: {str(output_file)}, with subsample_skip = {subsample_skip:d}")
    ds = ds.isel(time=slice(None, None, subsample_skip))
    ds.to_netcdf(output_file, engine="netcdf4")
    output_dict_subsample[component_name] = ds

## Visualization

In [ ]:
import matplotlib as mplt
if not use_ipython:
    mplt.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
from matplotlib.animation import FuncAnimation
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
import numpy as np

output_dict_animation = {
    component_name: _ds.isel(time=slice(None, None, 1))
    for component_name, _ds in output_dict_subsample.items()
}

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.gridlines(draw_labels=True)
cb = None
cf = None
cs = None
ch = None

def update(frame):
    print(f"Plotting frame={frame:d}")
    global cf, cb, cs, ch
    _data_q = output_dict_animation["atm"]["specific_humidity"].isel(time=frame, level=0)
    _data_sst = output_dict_animation["ocn"]["sea_surface_temperature"].isel(time=frame) - 273.15
    _data_sit = output_dict_animation["seaice"]["ice_thickness"].isel(time=frame)
    coords = _data_q.coords
    time_str = _data_q['time'].dt.strftime('%Y-%m-%d').to_numpy().item()
    lat = coords["lat"]
    lon = coords["lon"]

    # Remove previous frame's artists before drawing the new ones
    cf and cf.remove()
    cs and cs.remove()
    ch and ch.remove()
    
    # Plot the humidity field for the current time step
    cyclic_data_q, cyclic_lon = add_cyclic_point(_data_q.to_numpy().transpose(), coord=lon)
    mappable = ax.contourf(
        cyclic_lon, lat,
        cyclic_data_q,
        levels=1 + np.linspace(0, 1, 21) * 10,
        transform=ccrs.PlateCarree(), 
        cmap='GnBu',
        extend="both",
    )
    
    cyclic_data_sst, cyclic_lon = add_cyclic_point(_data_sst.to_numpy().transpose(), coord=lon)
    cs = ax.contour(
        cyclic_lon, lat,
        cyclic_data_sst,
        levels=np.arange(-2, 31, 4),
        transform=ccrs.PlateCarree(),
        colors="black",
    )
    ax.clabel(cs, fontsize=12)

    # Dot-hatch grid cells that carry any sea ice (thickness above zero)
    cyclic_data_sit, cyclic_lon = add_cyclic_point(_data_sit.to_numpy().transpose(), coord=lon)
    ch = ax.contourf(
        cyclic_lon, lat,
        cyclic_data_sit,
        levels=[1e-6, np.inf],
        colors="none",
        hatches=["."],
        transform=ccrs.PlateCarree(),
    )

    ax.set_title(f"[{time_str:s}]\nSurface specific humidity (shading) and sea surface temperature (contours, ${{}}^\\circ \\mathrm{{C}}$),\nwith sea ice (dotted hatching)")
    if cb is None:
        cb = plt.colorbar(ax=ax, mappable=mappable, orientation='vertical', shrink=0.7, pad=0.07)
        cb.set_label("[g/kg]", fontsize=12)
    
    return [cf,]
    
# Generate and save
ani = FuncAnimation(fig, update, frames=len(output_dict_animation["atm"].coords["time"]), interval=120, blit=False)
print("Saving animation: ", output_figures["animation"])
ani.save(output_figures["animation"], writer='pillow', dpi=200)

if use_ipython:
    from IPython.display import Image
    display(Image(output_figures["animation"]))